# 03 · DentVoxel 3D CBCT WSP Benchmark (Validation Only)

Evaluates **Track 1 (`dentvoxel_cls`: scan-split axial maxillary sinus presence, `cls-lenet2d-v1`)** and **Track 2 (`dentvoxel_seg`: 3D tooth union, `cbct-wsp82-v1`, 82 params)** directly from `DentVoxel.zip` (100 calibrated-HU NIfTI scans: **60 train / 20 val / 20 sealed test**).

In [ ]:
from pathlib import Path
import json
import sys

if Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive", readonly=True)
    DATA_SRC = Path("/content/drive/MyDrive/WSP_Dental_Data/DentVoxel.zip")
    candidates = [Path("/content/dental-wsp"), Path("/content/drive/MyDrive/dental-wsp")]
    REPO_DIR = next((p for p in candidates if (p / "benchmarks").is_dir()), candidates[1])
    OUT_ROOT = Path("/content/runs")
else:
    REPO_DIR = Path(r"C:\Users\Lenovo\Desktop\dental-wsp")
    DATA_SRC = Path(r"C:\Users\Lenovo\Desktop\WSP_Dental_Data\DentVoxel.zip")
    OUT_ROOT = REPO_DIR / "runs"

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from benchmarks.dentvoxel.adapter import make_dentvoxel_scan_splits, verify_dentvoxel_directory
from wsp_core.guards import exposure_guard

exposure_guard(DATA_SRC.parent)
audit = verify_dentvoxel_directory(DATA_SRC, max_scans=3)
print(f"Device: {DEVICE}")
print(json.dumps(audit, indent=2))

SPLIT_FILE = OUT_ROOT / "dentvoxel_splits.private.json"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
if not SPLIT_FILE.is_file():
    scan_ids = [f"scan_{i:04d}" for i in range(1, 101)]
    SPLIT_FILE.write_text(json.dumps(make_dentvoxel_scan_splits(scan_ids, seed=2026), indent=2), encoding="utf-8")
print(f"Deterministic 60/20/20 scan split ready at {SPLIT_FILE}")

In [ ]:
# Track 1 · dentvoxel_cls (Scan-Split Axial Maxillary Sinus Presence) — Stage 1, 2 & 3
import matplotlib.pyplot as plt
import pandas as pd
from benchmarks.dentvoxel.adapter import load_dentvoxel_cls
from wsp_core.training.runner import plot_classification_learning_curve, run_classification_three_arm_validation

cls_design = REPO_DIR / "benchmarks/dentvoxel/locks/dentvoxel_cls.design.lock.json"
cls_resp = REPO_DIR / "benchmarks/dentvoxel/locks/dentvoxel_cls.responses.lock.json"

if cls_resp.is_file():
    train_cls = load_dentvoxel_cls(DATA_SRC, SPLIT_FILE, split="train", norm_mode="hu_window")
    val_cls = load_dentvoxel_cls(DATA_SRC, SPLIT_FILE, split="val", norm_mode="hu_window")
    _, cls_summary = run_classification_three_arm_validation(
        task="dentvoxel_cls",
        train_samples=train_cls,
        val_samples=val_cls,
        design_lock_path=cls_design,
        responses_lock_path=cls_resp,
        budgets=[2, 4, 8, 16, 32, 60],
        seeds=[0, 1, 2, 3, 4],
        epochs=200,
        lr=5e-3,
        device=DEVICE,
        out_dir=OUT_ROOT / "dentvoxel_cls",
    )
    display(cls_summary.round(4))
    fig = plot_classification_learning_curve(cls_summary, task="dentvoxel_cls")
    plt.show()
    plt.close(fig)
else:
    print("Waiting for dentvoxel_cls.responses.lock.json")

In [ ]:
# Track 2 · dentvoxel_seg (3D CBCT Tooth Union, 82 params) — Stage 1, 2 & 3
from benchmarks.dentvoxel.adapter import load_dentvoxel_seg
from wsp_core.evaluation.segmentation import evaluate_segmentation_paired_audit, evaluate_segmentation_swaps
from wsp_core.schemas import strict_json_loads, verify_response_lock
from wsp_core.training.runner import plot_segmentation_learning_curve, run_segmentation_three_arm_validation

seg_design = REPO_DIR / "benchmarks/dentvoxel/locks/dentvoxel_seg.design.lock.json"
seg_resp = REPO_DIR / "benchmarks/dentvoxel/locks/dentvoxel_seg.responses.lock.json"

if seg_resp.is_file():
    val_seg = load_dentvoxel_seg(DATA_SRC, SPLIT_FILE, split="val", norm_mode="hu_window")
    d_bytes = seg_design.read_bytes()
    frozen_seg = verify_response_lock(strict_json_loads(d_bytes), d_bytes, strict_json_loads(seg_resp.read_bytes()), "cbct-wsp82-v1")
    rows = []
    for slot, payload in frozen_seg.items():
        sw = evaluate_segmentation_swaps(payload, val_seg)
        au = evaluate_segmentation_paired_audit(payload, val_seg)
        rows.append({
            "Reply": slot,
            "Intact Dice (%)": sw["full_cohort_dice_a_pct"],
            "Image-only Dice (%)": sw["full_cohort_dice_b_pct"],
            "Position-only Dice (%)": sw["full_cohort_dice_c_pct"],
            "Swap loss (pp)": sw["intact"]["swap_loss_pp"],
            "Outside FP share (%)": au["outside_fp_reduction_share_pct"],
            "3D Box IoU A (%)": au["box_iou_a_pct"],
            "Excluded tooth voxels A (%)": au["excluded_tooth_a_pct"],
        })
    display(pd.DataFrame(rows).set_index("Reply").round(4))

    train_seg = load_dentvoxel_seg(DATA_SRC, SPLIT_FILE, split="train", norm_mode="hu_window")
    _, seg_summary = run_segmentation_three_arm_validation(
        task="dentvoxel_seg",
        architecture="cbct-wsp82-v1",
        train_samples=train_seg,
        val_samples=val_seg,
        design_lock_path=seg_design,
        responses_lock_path=seg_resp,
        budgets=[1, 2, 4, 8, 16, 32, 60],
        seeds=[0, 1, 2],
        epochs=150,
        lr=1e-2,
        device=DEVICE,
        out_dir=OUT_ROOT / "dentvoxel_seg",
    )
    display(seg_summary.round(4))
    fig = plot_segmentation_learning_curve(seg_summary, task="dentvoxel_seg")
    plt.show()
    plt.close(fig)
else:
    print("Waiting for dentvoxel_seg.responses.lock.json")